# Cosmos3-Edge Policy with TensorRT-Edge-LLM

This notebook runs **Cosmos3-Edge-Policy-DROID** with
[TensorRT-Edge-LLM](https://github.com/NVIDIA/TensorRT-Edge-LLM) (v0.10.0+):
export the policy checkpoint, build the experimental Cosmos3 policy engines,
then run `cosmos3_policy_inference`.

Policy sits under the **Generator / Action** cookbook surface (action chunks),
not the Reasoner text path. For Edge multimodal reasoning on the same runtime,
see [`../../reasoner/run_with_trt_edge_llm.ipynb`](../../reasoner/run_with_trt_edge_llm.ipynb).

1. Points at a TensorRT-Edge-LLM checkout built with
   `-DBUILD_EXPERIMENTAL_MODELS=ON`.
2. Exports `nvidia/Cosmos3-Edge-Policy-DROID` with `--task policy`.
3. Builds policy engines.
4. Builds a 640×540 DROID `concat_view` from `assets/droid_lerobot_example`
   (same layout as the vLLM-Omni / SGLang policy notebooks) and writes
   `action.json`.

Complete the upstream
[Installation](https://nvidia.github.io/TensorRT-Edge-LLM/user_guide/getting_started/installation.html)
first. Command reference:
[Cosmos3-Edge VLA guide](https://nvidia.github.io/TensorRT-Edge-LLM/user_guide/examples/vla/cosmos3.html).


## 1. Setup paths

`EDGELLM_ROOT` must contain `build/experimental_models/cosmos3/examples/` after
a CMake build with `-DBUILD_EXPERIMENTAL_MODELS=ON`.


In [ ]:
from pathlib import Path
import os


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "cookbooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate the cosmos repository root")


COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
DROID_ASSET_ROOT = (
    COSMOS_ROOT / "cookbooks" / "cosmos3" / "generator" / "action" / "assets" / "droid_lerobot_example"
).resolve()
assert DROID_ASSET_ROOT.exists(), DROID_ASSET_ROOT

EDGELLM_ROOT = Path(
    os.environ.get("EDGELLM_ROOT", Path.home() / "TensorRT-Edge-LLM")
).expanduser().resolve()
ONNX_DIR = Path(
    os.environ.get(
        "ONNX_DIR",
        Path.home() / "tensorrt-edgellm-workspace" / "Cosmos3-Edge" / "onnx",
    )
).expanduser().resolve()
ENGINE_DIR = Path(
    os.environ.get(
        "ENGINE_DIR",
        Path.home() / "tensorrt-edgellm-workspace" / "Cosmos3-Edge" / "engines",
    )
).expanduser().resolve()
POLICY_CHECKPOINT = os.environ.get(
    "POLICY_CHECKPOINT", "nvidia/Cosmos3-Edge-Policy-DROID"
)
POLICY_PROMPT = os.environ.get(
    "POLICY_PROMPT",
    "Pick up the object and place it in the target container.",
)
WORK_DIR = ENGINE_DIR / "policy" / "cookbook_work"
WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON = WORK_DIR / "action.json"

os.environ["COSMOS_ROOT"] = str(COSMOS_ROOT)
os.environ["EDGELLM_ROOT"] = str(EDGELLM_ROOT)
os.environ["ONNX_DIR"] = str(ONNX_DIR)
os.environ["ENGINE_DIR"] = str(ENGINE_DIR)
os.environ["POLICY_CHECKPOINT"] = POLICY_CHECKPOINT
os.environ["DROID_ASSET_ROOT"] = str(DROID_ASSET_ROOT)
os.environ["POLICY_PROMPT"] = POLICY_PROMPT
os.environ["OUTPUT_JSON"] = str(OUTPUT_JSON)

print("cosmos root:", COSMOS_ROOT)
print("EDGELLM_ROOT:", EDGELLM_ROOT, "(exists:", EDGELLM_ROOT.exists(), ")")
print("ONNX_DIR:", ONNX_DIR)
print("ENGINE_DIR:", ENGINE_DIR)
print("POLICY_CHECKPOINT:", POLICY_CHECKPOINT)
print("DROID_ASSET_ROOT:", DROID_ASSET_ROOT)
print("prompt:", POLICY_PROMPT)


## 2. Export the Policy on CPU

Requires the TensorRT-Edge-LLM Python package (`tensorrt-edgellm-export`).


In [ ]:
%%bash
set -euo pipefail
: "${POLICY_CHECKPOINT:?run the setup cell first}"
: "${ONNX_DIR:?run the setup cell first}"

mkdir -p "$ONNX_DIR"
tensorrt-edgellm-export \
  "$POLICY_CHECKPOINT" \
  "$ONNX_DIR" \
  --task policy

echo "Exported policy ONNX under $ONNX_DIR"


## 3. Build all policy engines

The C++ runtime must have been configured with
`-DBUILD_EXPERIMENTAL_MODELS=ON`. Use `--maxBatchSize N` when the runtime must
accept more than one prompt.


In [ ]:
%%bash
set -euo pipefail
: "${EDGELLM_ROOT:?run the setup cell first}"
: "${ONNX_DIR:?run the setup cell first}"
: "${ENGINE_DIR:?run the setup cell first}"

cd "$EDGELLM_ROOT"
mkdir -p "$ENGINE_DIR"

./build/experimental_models/cosmos3/examples/cosmos3_policy_build \
  --onnxDir "$ONNX_DIR" \
  --engineDir "$ENGINE_DIR"

echo "Built policy engines under $ENGINE_DIR"


## 4. Prepare a DROID observation

This cell extracts the first frame from each checked-in DROID camera video and
builds the 640×540 `concat_view` used by the vLLM-Omni / SGLang policy
notebooks (wrist on top, two exterior views below). Pass that composite to
`--image`. The Edge-LLM runtime still resizes internally (736×544 in the
model contract); the **input** should be the DROID multiview, not a reasoner
caption asset.

Override `POLICY_PROMPT` in the setup cell if your instruction differs. The
upstream VLA guide's banana/bowl string is only a CLI placeholder.


In [ ]:
import subprocess

from PIL import Image

try:
    import imageio_ffmpeg
except ImportError as exc:
    raise RuntimeError(
        "Install imageio-ffmpeg in this notebook kernel: pip install imageio-ffmpeg"
    ) from exc

FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
CAMERA_VIDEO_PATHS = {
    "observation/wrist_image_left": DROID_ASSET_ROOT
    / "videos"
    / "observation.image.wrist_image_left"
    / "chunk-000"
    / "file-000.mp4",
    "observation/exterior_image_1_left": DROID_ASSET_ROOT
    / "videos"
    / "observation.image.exterior_image_1_left"
    / "chunk-000"
    / "file-000.mp4",
    "observation/exterior_image_2_left": DROID_ASSET_ROOT
    / "videos"
    / "observation.image.exterior_image_2_left"
    / "chunk-000"
    / "file-000.mp4",
}
for key, video_path in CAMERA_VIDEO_PATHS.items():
    assert video_path.exists(), f"missing {key}: {video_path}"


def extract_first_frame(video_path: Path, out_path: Path) -> Path:
    if not out_path.exists() or out_path.stat().st_mtime < video_path.stat().st_mtime:
        subprocess.run(
            [
                FFMPEG,
                "-y",
                "-loglevel",
                "error",
                "-i",
                str(video_path),
                "-frames:v",
                "1",
                str(out_path),
            ],
            check=True,
        )
    return out_path


frame_paths = {
    key: extract_first_frame(video_path, WORK_DIR / f"policy_{key.split('/')[-1]}.png")
    for key, video_path in CAMERA_VIDEO_PATHS.items()
}
frames = {key: Image.open(path).convert("RGB") for key, path in frame_paths.items()}

# Same geometry as run_policy_with_vllm_omni.ipynb / run_policy_with_sglang.ipynb:
# native 640×360 wrist on top; exteriors resized to 320×180 on the bottom row.
wrist = frames["observation/wrist_image_left"]
source_size = wrist.size
assert all(frame.size == source_size for frame in frames.values()), (
    "DROID camera frames must have matching dimensions",
    {key: frame.size for key, frame in frames.items()},
)
target_w = wrist.width
bottom_h, half_w = wrist.height // 2, wrist.width // 2
target_h = wrist.height + bottom_h
left = frames["observation/exterior_image_1_left"].resize(
    (half_w, bottom_h), Image.Resampling.BILINEAR
)
right = frames["observation/exterior_image_2_left"].resize(
    (half_w, bottom_h), Image.Resampling.BILINEAR
)
policy_image = Image.new("RGB", (target_w, target_h))
policy_image.paste(wrist, (0, 0))
policy_image.paste(left, (0, wrist.height))
policy_image.paste(right, (half_w, wrist.height))

OBSERVATION_IMAGE = WORK_DIR / "droid_policy_first_frame.png"
policy_image.save(OBSERVATION_IMAGE)
os.environ["OBSERVATION_IMAGE"] = str(OBSERVATION_IMAGE)

print("observation:", OBSERVATION_IMAGE, policy_image.size)
print("prompt:", POLICY_PROMPT)
try:
    from IPython.display import display

    display(policy_image)
except Exception:
    pass


## 5. Run the policy

`--image` is the 640×540 DROID concat from the previous cell. For an ordered
frame list, replace `--image` with
`--video frame_00.png,frame_01.png,frame_02.png`. The current policy conditions
on the most recent frame. Optional: `--steps` (denoise steps) and `--seed`.


In [ ]:
%%bash
set -euo pipefail
: "${EDGELLM_ROOT:?run the setup cell first}"
: "${ENGINE_DIR:?run the setup cell first}"
: "${OBSERVATION_IMAGE:?run the DROID observation cell first}"
: "${POLICY_PROMPT:?run the setup cell first}"
: "${OUTPUT_JSON:?run the setup cell first}"

cd "$EDGELLM_ROOT"
./build/experimental_models/cosmos3/examples/cosmos3_policy_inference \
  --engineDir "$ENGINE_DIR" \
  --image "$OBSERVATION_IMAGE" \
  --prompt "$POLICY_PROMPT" \
  --output "$OUTPUT_JSON"

echo "Wrote $OUTPUT_JSON"
python3 - <<'PY'
import json
import os
from pathlib import Path
data = json.loads(Path(os.environ["OUTPUT_JSON"]).read_text())
print(json.dumps(data, indent=2)[:4000])
PY


## 6. Next steps

- Override `POLICY_PROMPT` in the setup cell, or rebuild the concat with your
  own DROID camera videos.
- Multimodal reasoning on TensorRT-Edge-LLM:
  [`../../reasoner/run_with_trt_edge_llm.ipynb`](../../reasoner/run_with_trt_edge_llm.ipynb).
- OpenAI-compatible policy serving on workstation GPUs:
  [`run_policy_with_vllm_omni.ipynb`](./run_policy_with_vllm_omni.ipynb).
- Cosmos Framework policy server:
  [`run_policy_with_cosmos_framework.md`](./run_policy_with_cosmos_framework.md).
